# 30_00 — Phase 3 Overview, Environment Validation, and Shared CNN Scratch Setup

## Goal
This notebook prepares the project for **Phase 3 — CNN Models Trained From Scratch**.

It does **not** train models.  
Instead, it verifies that all required inputs, paths, configurations, and shared utilities are ready for the Phase 3 training notebooks:

- `30_01_customcnn_v1.ipynb`
- `30_02_customcnn_v2.ipynb`

## What this notebook validates
- project root resolution
- required Phase 1 assets (`split_v1`, `transforms_v1.yaml`)
- report/model/output directories
- MLflow tracking setup
- class mapping and split loading
- portable CSV filepath normalization
- train/eval transform loading
- dataset + dataloader sanity checks
- device detection
- one-batch shape checks

## Design Principles
This notebook is designed to be:

- **Run-All safe**
- **deterministic where appropriate**
- **portable across devices**
- **consistent with previous project pathing**
- **non-destructive** (no training, no overwrite of experiment runs)

## Fixed Contracts for Phase 3
- split: `split_v1`
- transforms config: `configs/transforms_v1.yaml`
- model family output root: `models/cnn_scratch/`
- reports root: `reports/`
- experiment tracking root: `mlruns/`

In [1]:
# Cell 2 — Imports

import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader

import mlflow

/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 3 — Project root + paths (same philosophy as Phase 2)

NOTEBOOK_DIR = Path.cwd().expanduser().resolve()

def find_project_root(start: Path) -> Path:
    markers_all = ["src", "configs"]
    markers_any = [".git", "pyproject.toml", "requirements.txt"]

    for p in [start] + list(start.parents):
        has_all = all((p / m).exists() for m in markers_all)
        has_any = any((p / m).exists() for m in markers_any)
        if has_all and has_any:
            return p

    for p in [start] + list(start.parents):
        if all((p / m).exists() for m in markers_all):
            return p

    raise RuntimeError(f"Could not locate project root from: {start}")

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
TRACKING_DIR = PROJECT_ROOT / "mlruns"

CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_ID = "split_v1"
SPLITS_DIR = DATA_DIR / "splits" / SPLIT_ID

TRAIN_CSV = SPLITS_DIR / "train.csv"
VAL_CSV = SPLITS_DIR / "val.csv"
TEST_CSV = SPLITS_DIR / "test.csv"
CLASSES_JSON = SPLITS_DIR / "classes.json"

TRANSFORMS_CONFIG_PATH = CONFIGS_DIR / "transforms_v1.yaml"
TRANSFORM_ID_TRAIN = "transforms_v1_train_runtime_aug"
TRANSFORM_ID_EVAL = "transforms_v1_eval_resize256_centercrop224_imagenetnorm"

CNN_SCRATCH_DIR = MODELS_DIR / "cnn_scratch"
CUSTOMCNN_V1_DIR = CNN_SCRATCH_DIR / "customcnn_v1"
CUSTOMCNN_V2_DIR = CNN_SCRATCH_DIR / "customcnn_v2"

REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
REPORTS_FIGURES_DIR = REPORTS_DIR / "figures"

CNN_SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
CUSTOMCNN_V1_DIR.mkdir(parents=True, exist_ok=True)
CUSTOMCNN_V2_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR       :", NOTEBOOK_DIR)
print("PROJECT_ROOT       :", PROJECT_ROOT)
print("SPLITS_DIR         :", SPLITS_DIR)
print("TRANSFORMS_CONFIG  :", TRANSFORMS_CONFIG_PATH)
print("CNN_SCRATCH_DIR    :", CNN_SCRATCH_DIR)
print("REPORTS_METRICS_DIR:", REPORTS_METRICS_DIR)
print("REPORTS_FIGURES_DIR:", REPORTS_FIGURES_DIR)
print("TRACKING_DIR       :", TRACKING_DIR)

NOTEBOOK_DIR       : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/notebooks/30_cnn_scratch_custom
PROJECT_ROOT       : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server
SPLITS_DIR         : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/data/splits/split_v1
TRANSFORMS_CONFIG  : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/configs/transforms_v1.yaml
CNN_SCRATCH_DIR    : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch
REPORTS_METRICS_DIR: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/reports/metrics
REPORTS_FIGURES_DIR: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/reports/figures
TRACKING_DIR       : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/mlruns


In [3]:
# Cell 4 — Add project root to Python path for src imports

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("[OK] PROJECT_ROOT added to sys.path")

[OK] PROJECT_ROOT added to sys.path


In [4]:
# Cell 5 — Required file existence checks

required_paths = {
    "train_csv": TRAIN_CSV,
    "val_csv": VAL_CSV,
    "test_csv": TEST_CSV,
    "classes_json": CLASSES_JSON,
    "transforms_yaml": TRANSFORMS_CONFIG_PATH,
}

missing_required = [name for name, path in required_paths.items() if not path.exists()]

if missing_required:
    details = "\n".join(f"- {name}: {required_paths[name]}" for name in missing_required)
    raise FileNotFoundError(
        "Phase 3 overview cannot continue because required files are missing:\n"
        f"{details}"
    )

print("[OK] All required Phase 1 configuration/split files exist.")
for name, path in required_paths.items():
    print(f"  - {name}: {path}")

[OK] All required Phase 1 configuration/split files exist.
  - train_csv: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/data/splits/split_v1/train.csv
  - val_csv: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/data/splits/split_v1/val.csv
  - test_csv: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/data/splits/split_v1/test.csv
  - classes_json: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/data/splits/split_v1/classes.json
  - transforms_yaml: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/configs/transforms_v1.yaml


In [5]:
# Cell 6 — MLflow setup (same project-level convention as Phase 2)

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

print("MLflow tracking URI:", TRACKING_DIR.as_uri())
print("MLflow experiment  : AnimalClassification")

MLflow tracking URI: file:///home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/mlruns
MLflow experiment  : AnimalClassification


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [6]:
# Cell 7 — Load split manifests and class mapping

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    class_to_idx = json.load(f)

idx_to_class = {v: k for k, v in class_to_idx.items()}

print("Loaded splits:")
print("  train:", len(train_df))
print("  val  :", len(val_df))
print("  test :", len(test_df))
print("Class mapping:", class_to_idx)

Loaded splits:
  train: 50127
  val  : 6266
  test : 6266
Class mapping: {'cats': 0, 'dogs': 1, 'wildlife': 2}


In [7]:
# Cell 8 — Portable filepath normalization (preserve Phase 2 logic)

def normalize_filepath(p: str) -> str:
    """
    Makes CSV filepaths portable across machines/OS.
    Handles:
      - Windows absolute paths containing /data/prepared/...
      - already-relative paths starting with data/...
      - backslashes -> slashes
    """
    p = str(p).strip()
    p2 = p.replace("\\", "/")

    marker = "/data/prepared/"
    low = p2.lower()
    if marker in low:
        idx = low.find(marker)
        rel = p2[idx + 1:]  # "data/prepared/..."
        return str((PROJECT_ROOT / rel).resolve())

    if p2.startswith("data/"):
        return str((PROJECT_ROOT / p2).resolve())

    return str(Path(p).expanduser().resolve()) if Path(p).exists() else p

for df in (train_df, val_df, test_df):
    df["filepath"] = df["filepath"].apply(normalize_filepath)

print("[OK] Filepaths normalized.")
print("Example normalized path:", train_df["filepath"].iloc[0] if len(train_df) else "N/A")

[OK] Filepaths normalized.
Example normalized path: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/data/prepared/wildlife/afd_DSC_1021_3.jpg


In [8]:
# Cell 9 — Add numeric labels and safely filter missing files before dataset loading

for df in (train_df, val_df, test_df):
    df["label_idx"] = df["label"].map(class_to_idx)
    assert df["label_idx"].isna().sum() == 0, "Found labels missing from classes.json mapping."

def filter_existing(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    exists = df["filepath"].apply(lambda p: Path(p).exists())
    missing_count = int((~exists).sum())
    if missing_count > 0:
        print(f"[WARNING] Dropping {missing_count} missing files from {split_name} split.")
    return df[exists].copy()

train_df = filter_existing(train_df, "train")
val_df = filter_existing(val_df, "val")
test_df = filter_existing(test_df, "test")

if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
    raise RuntimeError("One or more filtered splits are empty after filepath validation.")

print("Filtered split sizes:")
print("  train:", len(train_df))
print("  val  :", len(val_df))
print("  test :", len(test_df))

Filtered split sizes:
  train: 50127
  val  : 6266
  test : 6266


In [9]:
# Cell 10 — Split/class-distribution sanity summary

def summarize_split(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    counts = df["label"].value_counts().sort_index()
    out = pd.DataFrame({
        "split": split_name,
        "class": counts.index,
        "count": counts.values,
        "ratio": counts.values / counts.values.sum()
    })
    return out

summary_df = pd.concat([
    summarize_split(train_df, "train"),
    summarize_split(val_df, "val"),
    summarize_split(test_df, "test"),
], ignore_index=True)

print(summary_df)

   split     class  count     ratio
0  train      cats  18954  0.378120
1  train      dogs  18315  0.365372
2  train  wildlife  12858  0.256508
3    val      cats   2369  0.378072
4    val      dogs   2290  0.365464
5    val  wildlife   1607  0.256463
6   test      cats   2370  0.378232
7   test      dogs   2289  0.365305
8   test  wildlife   1607  0.256463


In [10]:
# Cell 11 — Load Phase 1 train/eval transforms

try:
    from src.data.transforms import (
        load_transforms_config,
        get_train_transforms,
        get_eval_transforms,
    )
except Exception as e:
    raise RuntimeError(
        "Could not import Phase 1 transforms from src.data.transforms.\n"
        "Make sure src/ exists and PROJECT_ROOT is correctly added to sys.path.\n"
        f"Original error: {e}"
    )

tfm_cfg = load_transforms_config(str(TRANSFORMS_CONFIG_PATH))
train_tfm = get_train_transforms(tfm_cfg)
eval_tfm = get_eval_transforms(tfm_cfg)

print("[OK] Loaded train/eval transforms from:", TRANSFORMS_CONFIG_PATH)
print("TRAIN TRANSFORM ID:", TRANSFORM_ID_TRAIN)
print("EVAL TRANSFORM ID :", TRANSFORM_ID_EVAL)

[OK] Loaded train/eval transforms from: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/configs/transforms_v1.yaml
TRAIN TRANSFORM ID: transforms_v1_train_runtime_aug
EVAL TRANSFORM ID : transforms_v1_eval_resize256_centercrop224_imagenetnorm


In [11]:
# Cell 12 — Try importing the Phase 1 dataset loader, with safe fallback if needed

DATASET_SOURCE = None

try:
    from src.data.dataset_loader import ImageDataset  # expected Phase 1 loader
    DATASET_SOURCE = "src.data.dataset_loader.ImageDataset"
    print(f"[OK] Using dataset loader from {DATASET_SOURCE}")
except Exception as e:
    ImageDataset = None
    print("[WARNING] Could not import ImageDataset from src.data.dataset_loader.")
    print("Original import error:", e)
    print("A notebook-local fallback dataset will be used for validation only.")

[OK] Using dataset loader from src.data.dataset_loader.ImageDataset


In [12]:
# Cell 13 — Dataset construction helpers (Run-All safe)

from torch.utils.data import Dataset

class CSVDatasetFallback(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.paths = self.df["filepath"].tolist()
        self.labels = self.df["label_idx"].astype(np.int64).tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx: int):
        from PIL import Image
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        y = self.labels[idx]
        return img, y

def build_dataset(df: pd.DataFrame, transform):
    """
    Uses the project dataset loader if available and compatible.
    Falls back to a local CSV dataset implementation otherwise.
    """
    if ImageDataset is not None:
        try:
            return ImageDataset(df=df.reset_index(drop=True).copy(), transform=transform)
        except TypeError:
            pass
        except Exception as e:
            print(f"[WARNING] ImageDataset(df=..., transform=...) failed: {e}")

        try:
            return ImageDataset(dataframe=df.reset_index(drop=True).copy(), transform=transform)
        except TypeError:
            pass
        except Exception as e:
            print(f"[WARNING] ImageDataset(dataframe=..., transform=...) failed: {e}")

        try:
            return ImageDataset(csv_df=df.reset_index(drop=True).copy(), transform=transform)
        except TypeError:
            pass
        except Exception as e:
            print(f"[WARNING] ImageDataset(csv_df=..., transform=...) failed: {e}")

    return CSVDatasetFallback(df=df, transform=transform)

train_ds = build_dataset(train_df, train_tfm)
val_ds = build_dataset(val_df, eval_tfm)

print("[OK] Dataset objects built.")
print("  train samples:", len(train_ds))
print("  val samples  :", len(val_ds))

[OK] Dataset objects built.
  train samples: 50127
  val samples  : 6266


In [13]:
# Cell 14 — Device and DataLoader setup

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = min(8, os.cpu_count() or 2)
PIN_MEMORY = True if DEVICE == "cuda" else False

# Conservative batch sizes for validation only in the overview notebook
BATCH_SIZE_TRAIN_SANITY = 32 if DEVICE == "cuda" else 16
BATCH_SIZE_EVAL_SANITY = 32 if DEVICE == "cuda" else 16

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE_TRAIN_SANITY,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE_EVAL_SANITY,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("DEVICE                 :", DEVICE)
print("NUM_WORKERS            :", NUM_WORKERS)
print("PIN_MEMORY             :", PIN_MEMORY)
print("BATCH_SIZE_TRAIN_SANITY:", BATCH_SIZE_TRAIN_SANITY)
print("BATCH_SIZE_EVAL_SANITY :", BATCH_SIZE_EVAL_SANITY)

DEVICE                 : cuda
NUM_WORKERS            : 8
PIN_MEMORY             : True
BATCH_SIZE_TRAIN_SANITY: 32
BATCH_SIZE_EVAL_SANITY : 32


In [14]:
# Cell 15 — One-batch sanity checks for train/val loaders

xb_train, yb_train = next(iter(train_loader))
xb_val, yb_val = next(iter(val_loader))

assert xb_train.ndim == 4, f"Expected 4D train batch tensor, got shape {tuple(xb_train.shape)}"
assert xb_val.ndim == 4, f"Expected 4D val batch tensor, got shape {tuple(xb_val.shape)}"

assert xb_train.shape[1] == 3, f"Expected RGB channels=3, got {xb_train.shape[1]}"
assert xb_val.shape[1] == 3, f"Expected RGB channels=3, got {xb_val.shape[1]}"

assert xb_train.shape[2] == 224 and xb_train.shape[3] == 224, (
    f"Expected train image size 224x224 after transforms, got {tuple(xb_train.shape[2:])}"
)
assert xb_val.shape[2] == 224 and xb_val.shape[3] == 224, (
    f"Expected val image size 224x224 after transforms, got {tuple(xb_val.shape[2:])}"
)

assert yb_train.ndim == 1, f"Expected 1D train labels, got shape {tuple(yb_train.shape)}"
assert yb_val.ndim == 1, f"Expected 1D val labels, got shape {tuple(yb_val.shape)}"

assert torch.isfinite(xb_train).all(), "Non-finite values found in train batch."
assert torch.isfinite(xb_val).all(), "Non-finite values found in val batch."

print("[OK] One-batch sanity checks passed.")
print("Train batch shape:", tuple(xb_train.shape), "Labels shape:", tuple(yb_train.shape))
print("Val batch shape  :", tuple(xb_val.shape), "Labels shape:", tuple(yb_val.shape))
print("Train labels unique in batch:", sorted(torch.unique(yb_train).tolist()))
print("Val labels unique in batch  :", sorted(torch.unique(yb_val).tolist()))

[OK] One-batch sanity checks passed.
Train batch shape: (32, 3, 224, 224) Labels shape: (32,)
Val batch shape  : (32, 3, 224, 224) Labels shape: (32,)
Train labels unique in batch: [0, 1, 2]
Val labels unique in batch  : [0, 1, 2]


In [15]:
# Cell 16 — Device transfer sanity check

xb_device = xb_train.to(DEVICE, non_blocking=True)
yb_device = yb_train.to(DEVICE, non_blocking=True)

print("[OK] Batch transfer to device succeeded.")
print("xb_device:", xb_device.shape, xb_device.device, xb_device.dtype)
print("yb_device:", yb_device.shape, yb_device.device, yb_device.dtype)

[OK] Batch transfer to device succeeded.
xb_device: torch.Size([32, 3, 224, 224]) cuda:0 torch.float32
yb_device: torch.Size([32]) cuda:0 torch.int64


In [16]:
# Cell 17 — Phase 3 run contract preview (paths only, no training yet)

PHASE3_RUN_CONTRACT = {
    "split_id": SPLIT_ID,
    "transforms_config": str(TRANSFORMS_CONFIG_PATH),
    "train_transform_id": TRANSFORM_ID_TRAIN,
    "eval_transform_id": TRANSFORM_ID_EVAL,
    "models_root": str(CNN_SCRATCH_DIR),
    "customcnn_v1_root": str(CUSTOMCNN_V1_DIR),
    "customcnn_v2_root": str(CUSTOMCNN_V2_DIR),
    "reports_metrics_dir": str(REPORTS_METRICS_DIR),
    "reports_figures_dir": str(REPORTS_FIGURES_DIR),
    "mlflow_tracking_dir": str(TRACKING_DIR),
    "required_run_artifacts": [
        "checkpoint.pt",
        "exported.onnx",
        "config.json",
        "metrics.json",
        "loss_curve.png",
        "accuracy_curve.png",
    ],
}

print(json.dumps(PHASE3_RUN_CONTRACT, indent=2))

{
  "split_id": "split_v1",
  "transforms_config": "/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/configs/transforms_v1.yaml",
  "train_transform_id": "transforms_v1_train_runtime_aug",
  "eval_transform_id": "transforms_v1_eval_resize256_centercrop224_imagenetnorm",
  "models_root": "/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch",
  "customcnn_v1_root": "/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v1",
  "customcnn_v2_root": "/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/cnn_scratch/customcnn_v2",
  "reports_metrics_dir": "/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/reports/metrics",
  "reports_figures_dir": "/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/reports/figures",
  "mlflow_tracking_

In [17]:
# Cell 18 — Save a small Phase 3 overview summary artifact to reports/metrics

overview_summary = {
    "phase": "phase_3_overview",
    "status": "ready",
    "project_root": str(PROJECT_ROOT),
    "split_id": SPLIT_ID,
    "train_count": int(len(train_df)),
    "val_count": int(len(val_df)),
    "test_count": int(len(test_df)),
    "class_to_idx": class_to_idx,
    "transforms_config": str(TRANSFORMS_CONFIG_PATH),
    "train_transform_id": TRANSFORM_ID_TRAIN,
    "eval_transform_id": TRANSFORM_ID_EVAL,
    "device_detected": DEVICE,
    "num_workers": int(NUM_WORKERS),
    "pin_memory": bool(PIN_MEMORY),
    "dataset_loader_source": DATASET_SOURCE or "fallback_csv_dataset",
    "phase3_model_dirs": {
        "customcnn_v1": str(CUSTOMCNN_V1_DIR),
        "customcnn_v2": str(CUSTOMCNN_V2_DIR),
    },
}

overview_summary_path = REPORTS_METRICS_DIR / "phase3_overview_summary.json"
tmp_path = overview_summary_path.with_suffix(".json.tmp")

with open(tmp_path, "w", encoding="utf-8") as f:
    json.dump(overview_summary, f, indent=2)

tmp_path.replace(overview_summary_path)

print("[OK] Saved overview summary to:", overview_summary_path)

[OK] Saved overview summary to: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/reports/metrics/phase3_overview_summary.json


In [18]:
# Cell 19 — MLflow logging for overview validation

with mlflow.start_run(run_name="phase3_overview_validation"):
    mlflow.log_param("stage", "phase3_overview")
    mlflow.log_param("split_id", SPLIT_ID)
    mlflow.log_param("train_transform_id", TRANSFORM_ID_TRAIN)
    mlflow.log_param("eval_transform_id", TRANSFORM_ID_EVAL)
    mlflow.log_param("device_detected", DEVICE)
    mlflow.log_param("num_workers", NUM_WORKERS)
    mlflow.log_param("pin_memory", PIN_MEMORY)
    mlflow.log_param("dataset_loader_source", DATASET_SOURCE or "fallback_csv_dataset")

    mlflow.log_metric("train_count", float(len(train_df)))
    mlflow.log_metric("val_count", float(len(val_df)))
    mlflow.log_metric("test_count", float(len(test_df)))
    mlflow.log_metric("train_batch_size_sanity", float(BATCH_SIZE_TRAIN_SANITY))
    mlflow.log_metric("eval_batch_size_sanity", float(BATCH_SIZE_EVAL_SANITY))

    mlflow.log_artifact(str(overview_summary_path))

print("[OK] MLflow overview validation logging complete.")

[OK] MLflow overview validation logging complete.


## Result

If all cells above completed successfully, Phase 3 prerequisites are in place:

- paths are resolved correctly
- required Phase 1 assets exist
- transforms load successfully
- dataset loading works
- dataloaders produce valid `(image_tensor, label)` batches
- device detection is working
- MLflow is configured
- model/report output roots exist

This notebook intentionally performs **no model training**.

The next notebooks should implement the actual scratch CNN experiments:

- `30_01_customcnn_v1.ipynb`
- `30_02_customcnn_v2.ipynb`